# NautilusTrader — Backtest Playground

EMA crossover strategy on EUR/USD tick data. Tweak parameters and re-run cells to experiment.

In [12]:
import urllib.request
from pathlib import Path

from nautilus_trader.backtest.engine import BacktestEngine, BacktestEngineConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import QuoteTickDataWrangler
from nautilus_trader.test_kit.providers import CSVTickDataLoader, TestInstrumentProvider

print("Imports OK ✓")

Imports OK ✓


## 1. Download & Prepare Data

In [13]:
url = "https://raw.githubusercontent.com/nautechsystems/nautilus_data/main/raw_data/fx_hist_data/DAT_ASCII_EURUSD_T_202001.csv.gz"
filename = "EURUSD_202001.csv.gz"

if not Path(filename).exists():
    print("Downloading sample tick data...")
    urllib.request.urlretrieve(url, filename)

instrument = TestInstrumentProvider.default_fx_ccy("EUR/USD")
wrangler = QuoteTickDataWrangler(instrument)

df = CSVTickDataLoader.load(filename, index_col=0, datetime_format="%Y%m%d %H%M%S%f")
df.columns = ["bid_price", "ask_price", "size"]
ticks = wrangler.process(df)

print(f"Loaded {len(ticks):,} ticks")
print(f"Period: {ticks[0].ts_init} → {ticks[-1].ts_init}")
df.head()

/Users/rc/Projects/workspace/.venv-nautilus/lib/python3.13/site-packages/nautilus_trader/persistence/loaders.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


Loaded 1,637,768 ticks
Period: 1577898010447000000 → 1580489996198000000


,bid_price,ask_price,size
20200101 170000065,,,
2020-01-01 17:00:10.447,1.12120,1.12192,0
2020-01-01 17:00:10.498,1.12117,1.12161,0
2020-01-01 17:00:12.579,1.12120,1.12161,0
2020-01-01 17:00:12.630,1.12120,1.12172,0
2020-01-01 17:00:12.839,1.12120,1.12171,0


## 2. Configure Strategy

**Tweak these parameters** and re-run to experiment:

In [14]:
from decimal import Decimal

from nautilus_trader.model.data import BarType

from strategies.forex.ema_cross import EMACrossConfig, EMACrossStrategy

# ═══════════════════════════════════════════
#  TWEAK THESE PARAMETERS
# ═══════════════════════════════════════════
FAST_EMA = 10       # Fast EMA period
SLOW_EMA = 20       # Slow EMA period
TRADE_SIZE = 100_000  # Position size (units)
BAR_INTERVAL = "1-MINUTE"  # Try: 1-MINUTE, 5-MINUTE, 15-MINUTE
# ═══════════════════════════════════════════

bar_type = BarType.from_str(f"EUR/USD.SIM-{BAR_INTERVAL}-MID-INTERNAL")

strategy_config = EMACrossConfig(
    instrument_id=instrument.id,
    bar_type=bar_type,
    trade_size=Decimal(TRADE_SIZE),
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
)

print(f"Strategy: EMA({FAST_EMA}/{SLOW_EMA}) on {BAR_INTERVAL} bars, size={TRADE_SIZE:,}")

Strategy: EMA(10/20) on 1-MINUTE bars, size=100,000


## 3. Run Backtest

In [15]:
# Build engine
engine = BacktestEngine(
    config=BacktestEngineConfig(
        logging=LoggingConfig(log_level="ERROR"),
    ),
)

# Add venue
engine.add_venue(
    venue=instrument.id.venue,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=USD,
    starting_balances=[Money(1_000_000, USD)],
)

# Add data & strategy
engine.add_instrument(instrument)
engine.add_data(ticks)
engine.add_strategy(EMACrossStrategy(config=strategy_config))

# Run
engine.run()
print("Backtest complete!")

Backtest complete!


## 4. Results & Analysis

In [16]:
# Account report
engine.trader.generate_account_report(instrument.id.venue)

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2020-01-01 17:00:10.447000+00:00,1000000.00,0.00,1000000.00,USD,SIM-001,MARGIN,USD,[],True,{}
2020-01-01 17:20:00+00:00,999997.76,336.45,999661.31,USD,SIM-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-01-01 17:59:00+00:00,999990.52,0.00,999990.52,USD,SIM-001,MARGIN,USD,[],False,{}
2020-01-01 17:59:00+00:00,999988.28,336.44,999651.84,USD,SIM-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-01-01 18:01:00+00:00,999960.04,0.00,999960.04,USD,SIM-001,MARGIN,USD,[],False,{}
...,...,...,...,...,...,...,...,...,...,...
2020-01-31 15:55:00+00:00,987675.96,0.00,987675.96,USD,SIM-001,MARGIN,USD,[],False,{}
2020-01-31 15:55:00+00:00,987673.74,332.65,987341.09,USD,SIM-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}
2020-01-31 15:59:00+00:00,987646.52,0.00,987646.52,USD,SIM-001,MARGIN,USD,[],False,{}
2020-01-31 15:59:00+00:00,987644.30,332.72,987311.58,USD,SIM-001,MARGIN,USD,"[{'type': 'MarginBalance', 'initial': '0.00', ...",False,{}


In [17]:
# Order fills
engine.trader.generate_order_fills_report()

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20200101-172000-001-000-1,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-001,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-001,MARKET,BUY,100000,...,None,None,None,None,None,None,None,3c711094-741d-4c22-a7d4-877b6686cf52,2020-01-01 17:20:00+00:00,2020-01-01 17:20:00+00:00
O-20200101-175900-001-000-2,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-002,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-002,MARKET,SELL,100000,...,None,None,None,None,None,None,None,bafc3085-f8ea-4ee8-b930-c16907a1c00c,2020-01-01 17:59:00+00:00,2020-01-01 17:59:00+00:00
O-20200101-175900-001-000-3,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-003,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-003,MARKET,SELL,100000,...,None,None,None,None,None,None,None,e138f49c-aec2-4780-afd4-cbf6f6c393d4,2020-01-01 17:59:00+00:00,2020-01-01 17:59:00+00:00
O-20200101-180100-001-000-4,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-004,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-004,MARKET,BUY,100000,...,None,None,None,None,None,None,None,74aaf4db-6179-46dd-b543-f4734c40edea,2020-01-01 18:01:00+00:00,2020-01-01 18:01:00+00:00
O-20200101-180100-001-000-5,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-005,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-005,MARKET,BUY,100000,...,None,None,None,None,None,None,None,1112200f-b1b4-4f48-ac7c-b09c1e5bd8c5,2020-01-01 18:01:00+00:00,2020-01-01 18:01:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O-20200131-155500-001-000-2862,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-2862,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-2862,MARKET,SELL,100000,...,None,None,None,None,None,None,None,33d7b59c-e69c-43c9-b3c6-88e95aaf6227,2020-01-31 15:55:00+00:00,2020-01-31 15:55:00+00:00
O-20200131-155500-001-000-2863,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-2863,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-2863,MARKET,SELL,100000,...,None,None,None,None,None,None,None,d55b6091-2708-4cbe-932a-5031f837e2fc,2020-01-31 15:55:00+00:00,2020-01-31 15:55:00+00:00
O-20200131-155900-001-000-2864,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-1-2864,EUR/USD.SIM-EMACrossStrategy-000,SIM-001,SIM-1-2864,MARKET,BUY,100000,...,None,None,None,None,None,None,None,54082db3-5500-484b-a766-e1bdb8d3b74a,2020-01-31 15:59:00+00:00,2020-01-31 15:59:00+00:00


In [18]:
# Position report
engine.trader.generate_positions_report()

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
EUR/USD.SIM-EMACrossStrategy-000-5e176d3c-60a3-4b43-9339-926a5019c8f1,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200101-172000-001-000-1,O-20200101-175900-001-000-2,BUY,FLAT,0,100000,...,2020-01-01 17:20:00+00:00,1577901540000000000,2020-01-01 17:59:00+00:00,2340000000000,1.12151,1.12146,[4.48 USD],-0.00004,-9.48 USD,True
EUR/USD.SIM-EMACrossStrategy-000-f765317c-a8ce-4753-90e0-100dd8b2b8fb,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200101-175900-001-000-3,O-20200101-180100-001-000-4,SELL,FLAT,0,100000,...,2020-01-01 17:59:00+00:00,1577901660000000000,2020-01-01 18:01:00+00:00,120000000000,1.12146,1.12172,[4.48 USD],-0.00023,-30.48 USD,True
EUR/USD.SIM-EMACrossStrategy-000-58765b10-538c-4c6d-880b-1501ba811d4a,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200101-180100-001-000-5,O-20200101-184700-001-000-6,BUY,FLAT,0,100000,...,2020-01-01 18:01:00+00:00,1577904420000000000,2020-01-01 18:47:00+00:00,2760000000000,1.12172,1.12192,[4.48 USD],0.00018,15.52 USD,True
EUR/USD.SIM-EMACrossStrategy-000-4fbd65cf-aa03-4f23-bfbe-f722dcc8248a,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200101-184700-001-000-7,O-20200101-194400-001-000-8,SELL,FLAT,0,100000,...,2020-01-01 18:47:00+00:00,1577907840000000000,2020-01-01 19:44:00+00:00,3420000000000,1.12192,1.12172,[4.48 USD],0.00018,15.52 USD,True
EUR/USD.SIM-EMACrossStrategy-000-37306a2d-3296-4575-8edb-8745a266109b,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200101-194400-001-000-9,O-20200101-204400-001-000-10,BUY,FLAT,0,100000,...,2020-01-01 19:44:00+00:00,1577911440000000000,2020-01-01 20:44:00+00:00,3600000000000,1.12172,1.12219,[4.48 USD],0.00042,42.52 USD,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EUR/USD.SIM-EMACrossStrategy-000-3f1646b1-0d40-419e-8748-58c868a69edd,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200131-145000-001-000-2857,O-20200131-150900-001-000-2858,BUY,FLAT,0,100000,...,2020-01-31 14:50:00+00:00,1580483340000000000,2020-01-31 15:09:00+00:00,1140000000000,1.10852,1.10859,[4.44 USD],0.00006,2.56 USD,True
EUR/USD.SIM-EMACrossStrategy-000-e8932735-f7da-4b2a-8f60-975c7f80ff01,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200131-150900-001-000-2859,O-20200131-151400-001-000-2860,SELL,FLAT,0,100000,...,2020-01-31 15:09:00+00:00,1580483640000000000,2020-01-31 15:14:00+00:00,300000000000,1.10859,1.10872,[4.44 USD],-0.00012,-17.44 USD,True
EUR/USD.SIM-EMACrossStrategy-000-c7aa1577-a33e-48fc-a62d-3abc5f2fa8c1,BACKTESTER-001,EMACrossStrategy-000,EUR/USD.SIM,SIM-001,O-20200131-151400-001-000-2861,O-20200131-155500-001-000-2862,BUY,FLAT,0,100000,...,2020-01-31 15:14:00+00:00,1580486100000000000,2020-01-31 15:55:00+00:00,2460000000000,1.10872,1.10882,[4.44 USD],0.00009,5.56 USD,True


## 5. Equity Curve

In [ ]:
import plotly.graph_objects as go

# Plot equity curve from position PnLs
positions = engine.trader.generate_positions_report()
if not positions.empty and "realized_pnl" in positions.columns:
    # realized_pnl is a string like "-9.48 USD" — extract the number
    pnl_values = positions["realized_pnl"].str.replace(r"\s+\w+$", "", regex=True).astype(float)
    cumulative_pnl = pnl_values.cumsum()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=cumulative_pnl.values,
        mode="lines",
        name="Cumulative PnL",
        line={"color": "cyan", "width": 2},
    ))
    fig.update_layout(
        title="Equity Curve (Cumulative PnL)",
        xaxis_title="Trade #",
        yaxis_title="PnL (USD)",
        template="plotly_dark",
        height=500,
    )
    fig.show()
else:
    print("No positions to plot")

In [ ]:
# PnL distribution
if not positions.empty and "realized_pnl" in positions.columns:
    pnls = positions["realized_pnl"].str.replace(r"\s+\w+$", "", regex=True).astype(float)
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=pnls.values,
        nbinsx=50,
        marker_color="cyan",
        opacity=0.7,
    ))
    fig.update_layout(
        title="PnL Distribution per Trade",
        xaxis_title="PnL (USD)",
        yaxis_title="Count",
        template="plotly_dark",
        height=400,
    )
    fig.show()


## 6. Cleanup

Reset the engine to run again with different parameters (go back to cell 2).

In [ ]:
engine.dispose()
print("Engine disposed — go back to cell 2 to tweak and re-run")